# 1. Modeling dataset

Build the product-week modeling dataset from `src.clean.load_clean`. Keep single-price weeks, flag and exclude the two partial weeks, and retain products with sufficient weekly observations and price variation. No model is fit in this section.

In [1]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
from IPython.display import display

project_root_candidates = (Path.cwd(), *Path.cwd().parents)
project_root = next(
    (candidate for candidate in project_root_candidates if (candidate / "src" / "clean.py").is_file()),
    None,
)
if project_root is None:
    raise FileNotFoundError("Could not locate the project root from the current directory.")
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.clean import load_clean

clean = load_clean(verbose=False)

# Match the product-week aggregation and modal-price tie-break in notebook 03.
product_week = (
    clean.groupby(["StockCode", "week"], as_index=False)
    .agg(
        units=("Quantity", "sum"),
        invoices=("InvoiceNo", "nunique"),
        n_prices=("UnitPrice", "nunique"),
    )
)
weekly_price_counts = (
    clean.groupby(["StockCode", "week", "UnitPrice"], as_index=False)
    .size()
    .rename(columns={"size": "price_frequency"})
    .sort_values(
        ["StockCode", "week", "price_frequency", "UnitPrice"],
        ascending=[True, True, False, True],
    )
    .drop_duplicates(["StockCode", "week"])
    .rename(columns={"UnitPrice": "price_mode"})
    [["StockCode", "week", "price_mode"]]
)
product_week = product_week.merge(
    weekly_price_counts, on=["StockCode", "week"], how="left", validate="one_to_one"
)[["StockCode", "week", "units", "invoices", "n_prices", "price_mode"]]
product_week = product_week.sort_values(["StockCode", "week"], ignore_index=True)

assert not product_week.duplicated(["StockCode", "week"]).any()
assert product_week["units"].sum() == clean["Quantity"].sum()

partial_week_starts = pd.to_datetime(["2010-11-29", "2011-12-05"])
product_week["is_partial_week"] = product_week["week"].isin(partial_week_starts)

def waterfall_row(step, frame):
    return {
        "step": step,
        "rows": len(frame),
        "products": frame["StockCode"].nunique(),
        "units": int(frame["units"].sum()),
    }

waterfall = [waterfall_row("Before filters", product_week)]
model_data = product_week.loc[product_week["n_prices"].eq(1)].copy()
assert int(model_data["units"].sum()) + int(
    product_week.loc[~product_week["n_prices"].eq(1), "units"].sum()
) == int(product_week["units"].sum())
waterfall.append(waterfall_row("1. Keep n_prices == 1", model_data))

before_partial_filter = model_data
model_data = model_data.loc[~model_data["is_partial_week"]].copy()
assert int(model_data["units"].sum()) + int(
    before_partial_filter.loc[before_partial_filter["is_partial_week"], "units"].sum()
) == int(before_partial_filter["units"].sum())
waterfall.append(waterfall_row("2. Drop flagged partial weeks", model_data))

product_eligibility = model_data.groupby("StockCode").agg(
    remaining_weeks=("week", "nunique"),
    distinct_prices=("price_mode", "nunique"),
)
eligible_products = product_eligibility.index[
    product_eligibility["remaining_weeks"].ge(10)
    & product_eligibility["distinct_prices"].ge(2)
]
before_product_filter = model_data
model_data = model_data.loc[model_data["StockCode"].isin(eligible_products)].copy()
removed_products = ~before_product_filter["StockCode"].isin(eligible_products)
assert int(model_data["units"].sum()) + int(
    before_product_filter.loc[removed_products, "units"].sum()
) == int(before_product_filter["units"].sum())
assert model_data.groupby("StockCode")["week"].nunique().ge(10).all()
assert model_data.groupby("StockCode")["price_mode"].nunique().ge(2).all()
waterfall.append(waterfall_row("3. Keep eligible products", model_data))

model_data["log_price"] = np.log(model_data["price_mode"])
model_data["log_units"] = np.log(model_data["units"])
model_data["log_invoices"] = np.log(model_data["invoices"])
log_columns = ["log_price", "log_units", "log_invoices"]
assert model_data[log_columns].notna().all().all()
assert np.isfinite(model_data[log_columns].to_numpy()).all()
assert int(model_data["units"].sum()) == int(
    waterfall[-1]["units"]
)
waterfall.append(waterfall_row("4. Add log columns (no filter)", model_data))

waterfall_table = pd.DataFrame(waterfall)
print("Unit reconciliation assertions: passed at every filtering step; log step preserves units.")
print("Log validity assertions: passed (no NaN or infinite values).")
display(waterfall_table)
print(
    f"Final sample: {len(model_data):,} product-weeks, "
    f"{model_data['StockCode'].nunique():,} products, "
    f"{int(model_data['units'].sum()):,} units; weeks "
    f"{model_data['week'].min():%Y-%m-%d} through {model_data['week'].max():%Y-%m-%d}."
)

Unit reconciliation assertions: passed at every filtering step; log step preserves units.
Log validity assertions: passed (no NaN or infinite values).


,step,rows,products,units
0,Before filters,100068,3786,5351288
1,1. Keep n_prices == 1,53956,3718,1083717
2,2. Drop flagged partial weeks,52023,3698,1048939
3,3. Keep eligible products,38653,1913,641536
4,4. Add log columns (no filter),38653,1913,641536


Final sample: 38,653 product-weeks, 1,913 products, 641,536 units; weeks 2010-12-06 through 2011-11-28.


## Final sample coverage

The final sample contains 38,653 observed product-weeks across 1,913 products, totaling 641,536 units; its week starts range from 2010-12-06 through 2011-11-28. It includes products with at least 10 remaining weeks and at least two distinct weekly modal prices, with exactly one recorded transaction price in each included product-week. The two flagged partial weeks (starting 2010-11-29 and 2011-12-05) are excluded. Missing product-weeks are not filled with zero sales. The sample does not cover mixed-price product-weeks, the excluded partial weeks, or products that fail the week-count or price-variation thresholds. No elasticity model is fit here.

## 2. Product-week diagnostics

Descriptive diagnostics for the complete, pre-filter product-week table; no model is fit.

In [2]:
line_counts = (
    clean.groupby(["StockCode", "week"])
    .size()
    .rename("lines")
    .reset_index()
)
product_week = product_week.merge(
    line_counts, on=["StockCode", "week"], how="left", validate="one_to_one"
)
assert product_week["lines"].notna().all()
assert product_week["lines"].sum() == len(clean)

product_week["price_mix"] = np.where(
    product_week["n_prices"].eq(1), "Single-price", "Mixed-price"
)
comparison_rows = []
for price_mix, group in product_week.groupby("price_mix", sort=False):
    row = {"price_mix": price_mix, "product_weeks": len(group)}
    for metric in ("lines", "invoices", "units"):
        row[f"{metric}_p25"] = group[metric].quantile(0.25)
        row[f"{metric}_median"] = group[metric].median()
        row[f"{metric}_p75"] = group[metric].quantile(0.75)
    comparison_rows.append(row)

price_mix_comparison = pd.DataFrame(comparison_rows)
print("Single-price vs mixed-price product-week distributions:")
display(price_mix_comparison)

Single-price vs mixed-price product-week distributions:


,price_mix,product_weeks,lines_p25,lines_median,lines_p75,invoices_p25,invoices_median,invoices_p75,units_p25,units_median,units_p75
0,Mixed-price,46112,4.0,6.0,11.0,4.0,6.0,10.0,14.0,38.0,101.0
1,Single-price,53956,1.0,1.0,3.0,1.0,1.0,3.0,2.0,6.0,20.0


In [3]:
invoice_bucket_labels = ["1", "2-3", "4-10", "11+"]
product_week["invoice_bucket"] = pd.cut(
    product_week["invoices"],
    bins=[0, 1, 3, 10, np.inf],
    labels=invoice_bucket_labels,
    right=True,
)
invoice_bucket_summary = (
    product_week.groupby("invoice_bucket", observed=False)
    .agg(
        product_weeks=("StockCode", "size"),
        single_price_weeks=("n_prices", lambda values: values.eq(1).sum()),
    )
    .reindex(invoice_bucket_labels)
    .reset_index()
)
invoice_bucket_summary["single_price_share_pct"] = (
    100 * invoice_bucket_summary["single_price_weeks"]
    / invoice_bucket_summary["product_weeks"]
)
assert invoice_bucket_summary["product_weeks"].sum() == len(product_week)
assert invoice_bucket_summary["single_price_weeks"].sum() == int(
    product_week["n_prices"].eq(1).sum()
)
print("Single-price share by product-week invoice bucket:")
display(invoice_bucket_summary)

Single-price share by product-week invoice bucket:


,invoice_bucket,product_weeks,single_price_weeks,single_price_share_pct
0,1,28085,28074,99.960833
1,2-3,27381,16388,59.851722
2,4-10,32475,8839,27.217860
3,11+,12127,655,5.401171


In [4]:
mode_observations = clean.merge(
    product_week[["StockCode", "week", "price_mode"]],
    on=["StockCode", "week"],
    how="left",
    validate="many_to_one",
)
mode_counts = (
    mode_observations.loc[mode_observations["UnitPrice"].eq(mode_observations["price_mode"])]
    .groupby(["StockCode", "week"])
    .agg(
        mode_lines=("UnitPrice", "size"),
        mode_units=("Quantity", "sum"),
    )
    .reset_index()
)
product_week = product_week.merge(
    mode_counts, on=["StockCode", "week"], how="left", validate="one_to_one"
)
product_week["mode_share_lines"] = product_week["mode_lines"] / product_week["lines"]
product_week["mode_share_units"] = product_week["mode_units"] / product_week["units"]
mode_share_columns = ["mode_share_lines", "mode_share_units"]
assert product_week[mode_share_columns].gt(0).all().all()
assert product_week[mode_share_columns].le(1).all().all()

all_product_weeks = len(product_week)
all_units = int(product_week["units"].sum())
threshold_rows = []
for metric in ("mode_share_units", "mode_share_lines"):
    for threshold in (0.9, 0.8):
        qualifying = product_week.loc[product_week[metric].ge(threshold)]
        threshold_rows.append(
            {
                "measure": metric,
                "threshold": f">= {threshold:.1f}",
                "product_weeks": len(qualifying),
                "share_all_rows_pct": 100 * len(qualifying) / all_product_weeks,
                "units": int(qualifying["units"].sum()),
                "share_all_units_pct": 100 * qualifying["units"].sum() / all_units,
            }
        )
mode_share_summary = pd.DataFrame(threshold_rows)
print("Product-weeks and units meeting modal-price concentration thresholds:")
display(mode_share_summary)

Product-weeks and units meeting modal-price concentration thresholds:


,measure,threshold,product_weeks,share_all_rows_pct,units,share_all_units_pct
0,mode_share_units,>= 0.9,68439,68.392493,2204492,41.195540
1,mode_share_units,>= 0.8,75882,75.830435,2518320,47.060072
2,mode_share_lines,>= 0.9,55945,55.906983,1390710,25.988323
3,mode_share_lines,>= 0.8,65400,65.355558,2429388,45.398192


## 3. Sample definitions

Define four product-week samples from the complete diagnostics table, excluding flagged partial weeks from every sample and denominator. No model is fit.

In [5]:
nonpartial_weeks = product_week.loc[~product_week["is_partial_week"]].copy()
assert "is_partial_week" in product_week.columns

sample_flags = {
    "A": nonpartial_weeks["n_prices"].eq(1),
    "B": nonpartial_weeks["invoices"].ge(3)
    & nonpartial_weeks["mode_share_lines"].ge(0.8),
    "C": nonpartial_weeks["invoices"].ge(2)
    & nonpartial_weeks["mode_share_units"].ge(0.9),
    "D": nonpartial_weeks["invoices"].ge(5)
    & nonpartial_weeks["mode_share_lines"].ge(0.8),
}
for name, flag in sample_flags.items():
    nonpartial_weeks[f"sample_{name}"] = flag

sample_a = nonpartial_weeks.loc[nonpartial_weeks["sample_A"]]
assert len(sample_a) == 52_023
assert int(sample_a["units"].sum()) == 1_048_939
sample_b = nonpartial_weeks.loc[nonpartial_weeks["sample_B"]]
assert sample_b["invoices"].ge(3).all()

denominator_product_weeks = len(nonpartial_weeks)
denominator_units = int(nonpartial_weeks["units"].sum())
sample_rows = []
for name in ("A", "B", "C", "D"):
    sample = nonpartial_weeks.loc[nonpartial_weeks[f"sample_{name}"]]
    log_values = pd.DataFrame(
        {
            "log_price": np.log(sample["price_mode"]),
            "log_units": np.log(sample["units"]),
            "log_invoices": np.log(sample["invoices"]),
        },
        index=sample.index,
    )
    assert log_values.notna().all().all(), f"NaN log value in sample {name}"
    assert np.isfinite(log_values.to_numpy()).all(), f"Infinite log value in sample {name}"

    product_eligibility = sample.groupby("StockCode").agg(
        weeks=("week", "nunique"),
        price_modes=("price_mode", "nunique"),
    )
    eligible_products = product_eligibility.index[
        product_eligibility["weeks"].ge(10)
        & product_eligibility["price_modes"].ge(2)
    ]
    eligible_sample = sample.loc[sample["StockCode"].isin(eligible_products)]
    sample_rows.append(
        {
            "sample": name,
            "product_weeks": len(sample),
            "products": sample["StockCode"].nunique(),
            "units": int(sample["units"].sum()),
            "share_all_product_weeks_pct": 100 * len(sample) / denominator_product_weeks,
            "share_all_units_pct": 100 * sample["units"].sum() / denominator_units,
            "eligible_products": len(eligible_products),
            "eligible_product_weeks": len(eligible_sample),
            "eligible_units": int(eligible_sample["units"].sum()),
            "invoices_eq_1_product_weeks": int(sample["invoices"].eq(1).sum()),
        }
    )

sample_summary = pd.DataFrame(sample_rows)
print("Sample definitions and coverage (partial weeks excluded):")
display(sample_summary)

Sample definitions and coverage (partial weeks excluded):


,sample,product_weeks,products,units,share_all_product_weeks_pct,share_all_units_pct,eligible_products,eligible_product_weeks,eligible_units,invoices_eq_1_product_weeks
0,A,52023,3698,1048939,54.295256,20.580421,1913,38653,641536,27107
1,B,25597,2724,1959089,26.715024,38.437771,348,5533,365340,0
2,C,38814,3218,1883439,40.509315,36.953501,1033,19606,855290,0
3,D,16891,2205,1699523,17.628764,33.345027,153,2326,226494,0
